In [8]:
import sqlite3
import pandas as pd
import numpy as np
import torch
import re
import json
from sentence_transformers import SentenceTransformer, util
from sklearn.cluster import AgglomerativeClustering
from IPython.display import display, HTML
from tqdm.notebook import tqdm

# --- 1. CONFIGURATION ---
DB_PATH = "db/results.sqlite"

# Detect Hardware
device = "cpu"
if torch.cuda.is_available(): device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available(): device = "mps"

print(f"⚡ Device: {device.upper()}")

# Load Model
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# Pastel Palette for Clusters
COLORS = [
    "#FFB3BA", "#BAFFC9", "#BAE1FF", "#FFFFBA", "#FFDFBA", 
    "#E0BBE4", "#957DAD", "#D291BC", "#FEC8D8", "#FFDFD3",
    "#A0E7E5", "#B4F8C8", "#FBE7C6", "#FFAEBC", "#D4F0F0"
]

# --- 2. SEGMENTATION & CLUSTERING HELPERS ---

def split_into_segments(text):
    """Splits text into reasoning steps, preserving decimals."""
    if not text: return []
    text = re.sub(r'(\d)\.(\d)', r'\1<DECIMAL>\2', text)
    parts = re.split(r'(\n|\. )', text)
    segments = []
    for p in parts:
        if not p.strip(): continue
        segments.append(p.replace("<DECIMAL>", "."))
    return [s for s in segments if s.strip()]

def get_cluster_colors(all_segments):
    """Clusters segments from ALL traces to find shared meanings."""
    if not all_segments: return {}, []
    
    embeddings = model.encode(all_segments, convert_to_tensor=True)
    embeddings_cpu = embeddings.cpu().numpy()
    embeddings_cpu = embeddings_cpu / np.linalg.norm(embeddings_cpu, axis=1, keepdims=True)
    
    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=0.4, 
        metric='euclidean',
        linkage='average'
    ).fit(embeddings_cpu)
    
    labels = clustering.labels_
    seg_colors = {}
    for i, label in enumerate(labels):
        seg_colors[i] = COLORS[label % len(COLORS)]
        
    return seg_colors, labels

# --- 3. VISUALIZATION GENERATOR ---

def render_comparison(question_data, traces, sim_matrix):
    """
    Generates the HTML Grid. 
    NOTE: Accepts pre-calculated sim_matrix to avoid re-running embeddings.
    """
    all_segments_flat = []
    trace_maps = [] 
    
    # Map segments for coloring
    for t in traces:
        segs = split_into_segments(t)
        seg_indices = []
        for s in segs:
            seg_indices.append(len(all_segments_flat))
            all_segments_flat.append(s)
        trace_maps.append(list(zip(segs, seg_indices)))
        
    # Cluster Segments
    color_map, cluster_ids = get_cluster_colors(all_segments_flat)
    
    # Build HTML
    html = f"""
    <div style='font-family: sans-serif; border: 1px solid #ccc; padding: 20px; border-radius: 8px; margin-bottom: 30px; background: #fdfdfd;'>
        <h3 style='margin-top:0'>[Index {question_data['idx']}] Q: {question_data['q'][:100]}...</h3>
        <p><strong>Gold:</strong> {question_data['gold']}</p>
        <div style='display: flex; gap: 15px; overflow-x: auto;'>
    """
    
    for i, (trace_txt, segments) in enumerate(zip(traces, trace_maps)):
        # Retrieve the pre-calculated vector for this trace
        sim_vector = sim_matrix[i]
        sim_str = "[" + ", ".join([f"{s:.2f}" for s in sim_vector]) + "]"
        
        html += f"""
        <div style='flex: 1; min-width: 300px; border: 1px solid #ddd; border-radius: 5px; background: #fff;'>
            <div style='background: #f8f9fa; padding: 10px; border-bottom: 1px solid #ddd; text-align: center;'>
                <strong>Generation {i+1}</strong><br>
                <div style='font-size: 0.85em; color: #555; margin-top:4px; font-family: monospace;'>
                    Sim: {sim_str}
                </div>
            </div>
            <div style='padding: 15px; line-height: 1.6; font-size: 0.9em;'>
        """
        
        for text, global_idx in segments:
            bg_color = color_map.get(global_idx, "#ffffff")
            # Safe get for cluster id in tooltip
            cid = cluster_ids[global_idx] if len(cluster_ids) > global_idx else "?"
            html += f"<span style='background-color: {bg_color}; padding: 2px 4px; margin: 0 1px; border-radius: 3px; display: inline-block; margin-bottom: 4px;' title='Cluster ID: {cid}'>{text}</span> "
            
        html += "</div></div>"
        
    html += "</div></div>"
    return html

# --- 4. MAIN PROCESSING LOOP ---

def process_and_update_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    # 1. Ensure DB has the column for the vectors
    try:
        cursor.execute("ALTER TABLE Results ADD COLUMN similarity_vector TEXT")
        conn.commit()
    except sqlite3.OperationalError:
        pass 

    query = """
    SELECT question_id_external, question_text, gold_answer 
    FROM Results 
    GROUP BY question_id_external 
    HAVING COUNT(*) >= 2
    """
    questions = conn.execute(query).fetchall()
    
    print(f"🚀 Starting processing for {len(questions)} unique questions...")
    
    for i, q in tqdm(enumerate(questions), total=len(questions), desc="Updating Traces"):
        
        # FIX: Use 'result_id' (explicit PK) instead of 'rowid'
        trace_rows = conn.execute(
            "SELECT result_id, full_trace_text FROM Results WHERE question_id_external = ?", 
            (q['question_id_external'],)
        ).fetchall()
        
        traces = [r['full_trace_text'] for r in trace_rows]
        # FIX: Access the explicit column name
        row_ids = [r['result_id'] for r in trace_rows] 
        
        # --- Embeddings & Sim Matrix ---
        full_trace_embs = model.encode(traces, convert_to_tensor=True)
        sim_matrix = util.cos_sim(full_trace_embs, full_trace_embs).cpu().numpy()
        
        for j, r_id in enumerate(row_ids):
            vector_list = sim_matrix[j].tolist()
            vector_json = json.dumps([round(x, 4) for x in vector_list])
            
            # FIX: Update using 'result_id'
            conn.execute(
                "UPDATE Results SET similarity_vector = ? WHERE result_id = ?",
                (vector_json, r_id)
            )
        
        # --- Visualization ---
        if i < 3 or i % 100 == 0:
            q_data = {'idx': i, 'q': q['question_text'], 'gold': q['gold_answer']}
            html_view = render_comparison(q_data, traces, sim_matrix)
            display(HTML(html_view))
            
        conn.commit()

    conn.close()
    print("✨ Processing Complete. Database updated.")

process_and_update_db()

# Run it
process_and_update_db()

⚡ Device: MPS
🚀 Starting processing for 1000 unique questions...


Updating Traces:   0%|          | 0/1000 [00:00<?, ?it/s]

✨ Processing Complete. Database updated.
🚀 Starting processing for 1000 unique questions...


Updating Traces:   0%|          | 0/1000 [00:00<?, ?it/s]

✨ Processing Complete. Database updated.
